In [12]:
import pandas as pd
import time
from nba_api.stats.endpoints import leaguedashplayerstats

In [2]:
seasons = []
# season_number = 3
season_number = 26

for i in range(season_number):
    if i < 9:  
        seasons.append(f'200{i}-0{i+1}')
    elif i==9:
        seasons.append(f'200{i}-{i+1}')
    else:
        seasons.append(f'20{i}-{i+1}')

print(seasons)

['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']


In [14]:
import os
os.makedirs('../data/raw', exist_ok=True) 

path_basic_allnba = '../data/raw/basic_allnba.csv'
path_adv_allnba = '../data/raw/adv_allnba.csv'

path_basic_rookie = '../data/raw/basic_rookie.csv'
path_adv_rookie = '../data/raw/adv_rookie.csv'

In [15]:
print('Zapis dla basic allnba')
basic_stats_allnba = []

if os.path.exists(path_basic_allnba):
    print(f'Plik istnieje: {path_basic_allnba}')
else:
    for name in seasons:
        try:
            stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=name, 
                season_type_all_star='Regular Season'
            )
            df = stats.get_data_frames()[0]
            df['SEASON'] = name 

            basic_stats_allnba.append(df)
            print('Download season:', name)
        except Exception as ex:
            print(f"{name} — błąd: {ex}")
        time.sleep(2)

    if len(basic_stats_allnba) == len(seasons):
        df_basic_allnba = pd.concat(basic_stats_allnba, ignore_index=True)
        df_basic_allnba.to_csv(path_basic_allnba, index=False)
        print('Zapisano do csv')

Zapis dla basic allnba
Plik istnieje: ../data/raw/basic_allnba.csv


In [16]:
print('Zapis dla advanced allnba')
adv_stats_allnba = []
failed_cols = []

if os.path.exists(path_adv_allnba):
    print(f'Plik istnieje:{path_adv_allnba}')
else:
    for name in seasons:
        try:
            stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=name,
                measure_type_detailed_defense='Advanced', 
                season_type_all_star='Regular Season',
                timeout=60
            )
            df = stats.get_data_frames()[0]
            df['SEASON'] = name 

            adv_stats_allnba.append(df)
            print('Download season:', name)
        except Exception as ex:
            print(f"{name} — błąd: {ex}")
            failed_cols.append(name)
        time.sleep(10)

    if len(adv_stats_allnba) == len(seasons):
        df_adv_allnba = pd.concat(adv_stats_allnba, ignore_index=True)
        df_adv_allnba.to_csv(path_adv_allnba, index=False)
        print('Zapisano do csv')

Zapis dla advanced allnba
Plik istnieje:../data/raw/adv_allnba.csv


In [17]:
print('Zapis dla basic rookies')
basic_stats_rookie = []

if os.path.exists(path_basic_rookie):
    print(f'Plik istnieje: {path_basic_rookie}')
else:
    for name in seasons:
        try:
            stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=name, 
                season_type_all_star='Regular Season',
                player_experience_nullable='Rookie', 
                timeout=60
            )
            df = stats.get_data_frames()[0]
            df['SEASON'] = name 

            basic_stats_rookie.append(df)
            print('Download season:', name)
        except Exception as ex:
            print(f"{name} — błąd: {ex}")
        time.sleep(5)

    if len(basic_stats_rookie) == len(seasons):
        df_basic_rookie = pd.concat(basic_stats_rookie, ignore_index=True)
        df_basic_rookie.to_csv(path_basic_rookie, index=False)
        print('Zapisano do csv')

Zapis dla basic rookies
Plik istnieje: ../data/raw/basic_rookie.csv


In [18]:
print('Zapis dla adv rookies')
adv_stats_rookie = []

if os.path.exists(path_adv_rookie):
    print(f'Plik istnieje: {path_adv_rookie}')
else:
    for name in seasons:
        try:
            stats = leaguedashplayerstats.LeagueDashPlayerStats(
                season=name, 
                season_type_all_star='Regular Season',
                measure_type_detailed_defense='Advanced', 
                player_experience_nullable='Rookie', 
                timeout=60
            )
            df = stats.get_data_frames()[0]
            df['SEASON'] = name 

            adv_stats_rookie.append(df)
            print('Download season:', name)
        except Exception as ex:
            print(f"{name} — błąd: {ex}")
        time.sleep(5)

    if len(adv_stats_rookie) == len(seasons):
        df_adv_rookie = pd.concat(adv_stats_rookie, ignore_index=True)
        df_adv_rookie.to_csv(path_adv_rookie, index=False)
        print('Zapisano do csv')

Zapis dla adv rookies
Plik istnieje: ../data/raw/adv_rookie.csv


In [19]:
path_id_allnba = '../data/raw/id_allnba.csv'
path_id_rookie = '../data/raw/id_rookie.csv'

id_cols = ['PLAYER_ID', 'PLAYER_NAME', 'SEASON', 'TEAM_ABBREVIATION']

In [20]:
def write_id(cols, path, input_df):
    print(f'Zapis id dla: {path}')
    df_name_id = pd.DataFrame()

    if os.path.exists(path):
        print(f'Plik istnieje: {path}')
    else:
        for col in cols:
            df_name_id[col] = input_df[col]
                
        if len(df_name_id) > 0:
            df_name_id.to_csv(path, index=False)
            print(f'Zapisano do csv:{path}')

In [21]:
# write_id(id_cols, path_id_allnba, df_basic_allnba)
# write_id(id_cols, path_id_rookie, df_basic_rookie)

In [22]:
from unidecode import unidecode
def process_raw_data(df):
    rows = []
    
    for _, row in df.iterrows():
        if pd.isna(row['Season']):
            continue
        
        season = row['Season']
        team_num = 1 if row['Tm'] == '1st' else 2
        
        player_cols = ['Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']
        
        for col in player_cols:
            player = row[col]
            if pd.notna(player):
                player_clean = unidecode(str(player).strip())
                player_clean = player_clean.rsplit(' ', 1)[0] if player_clean[-1] in ['C', 'F', 'G'] else player_clean
                
                rows.append({
                    'PLAYER_NAME': player_clean,
                    'SEASON': season,
                    'TEAM': team_num
                })
    
    return pd.DataFrame(rows)  


In [23]:
path_voting_allnba = '../data/raw/voting_allnba.csv'
path_voting_rookie = '../data/raw/voting_rookie.csv'

In [ ]:
import time
import pandas as pd
import os
years = range(2001, 2026)  


voting_allnba = []
voting_rookie = []
cols_voting = ['Player', 'Pts Won', 'Pts Max', 'SEASON']

num_of_rows_allnba = 18
num_of_rows_rookie = 11

if os.path.exists(path_voting_allnba) and os.path.exists(path_voting_rookie):
    print(f'Plik istnieje: {path_voting_allnba} i {path_voting_rookie}')
else:
    for year in years:
        try:
            url = f'https://www.basketball-reference.com/awards/awards_{year}.html'
            tables = pd.read_html(url)

            if len(tables) == 5:
                # -----ALL-NBA------
                df_nba = tables[2].copy()
                df_nba.columns = df_nba.columns.droplevel(0)
                df_nba['SEASON'] = f'{year-1}-{str(year)[-2:]}'
                
                df_nba = df_nba[cols_voting]
                df_nba_cut = df_nba.head(num_of_rows_allnba)
                # print(df_nba_cut)
                df_nba_cut = df_nba_cut.dropna().reset_index(drop=True)
                voting_allnba.append(df_nba_cut)
                # print(df_nba_cut)

                # -----ROOKIE------
                df_rookie = tables[4].copy()
                df_rookie.columns = df_rookie.columns.droplevel(0)
                df_rookie['SEASON'] = f'{year-1}-{str(year)[-2:]}'

                df_rookie = df_rookie[cols_voting]
                df_rookie_cut = df_rookie.head(num_of_rows_rookie)
                df_rookie_cut = df_rookie_cut.dropna().reset_index(drop=True)
                voting_rookie.append(df_rookie_cut)
                # print(df_nba_cut.shape)
                # print(df_rookie_cut.shape)

                print(f'Downloada year: {year}')
        

        except Exception as ex:
            print(f'{year} - {ex}')

        time.sleep(5)


    if len(voting_allnba) == len(years):
        df_voting_allnba = pd.concat(voting_allnba, ignore_index=True)
        df_voting_allnba.to_csv(path_voting_allnba, index=False)
        print('Zapisano do csv allnba')


    if len(voting_rookie) == len(years):
        df_voting_rookie = pd.concat(voting_rookie, ignore_index=True)
        df_voting_rookie.to_csv(path_voting_rookie, index=False)
        print('Zapisano do csv rookie')

#### Dodanie statystyk drużyny 


In [ ]:
from nba_api.stats.endpoints import leaguedashteamstats

path_team_allnba = '../data/raw/team_allnba.csv'
path_team_rookie = '../data/raw/team_rookie.csv'

In [10]:
print('Zapis dla team allnba')
team_stats_allnba = []
failed_cols = []

if os.path.exists(path_team_allnba):
    print(f'Plik istnieje:{path_team_allnba}')
else:
    for name in seasons:
        try:
            stats = leaguedashteamstats.LeagueDashTeamStats(
                season=name,
                season_type_all_star='Regular Season', 
                timeout=60
            )
            df = stats.get_data_frames()[0]
            df['SEASON'] = name 

            team_stats_allnba.append(df)
            print('Download season:', name)
        except Exception as ex:
            print(f"{name} — błąd: {ex}")
            failed_cols.append(name)
        time.sleep(10)

    if len(team_stats_allnba) == len(seasons):
        df_team_stats = pd.concat(team_stats_allnba, ignore_index=True)
        df_team_stats.to_csv(path_team_allnba, index=False)
        print('Zapisano do csv')

Zapis dla team allnba
Plik istnieje:../data/raw/team_allnba.csv


In [13]:
print('Zapis dla team rookie')
team_stats_rookie = []
failed_cols = []

if os.path.exists(path_team_rookie):
    print(f'Plik istnieje:{path_team_rookie}')
else:
    for name in seasons:
        try:
            stats = leaguedashteamstats.LeagueDashTeamStats(
                season=name,
                season_type_all_star='Regular Season', 
                player_experience_nullable='Rookie', 
                timeout=60
            )
            df = stats.get_data_frames()[0]
            df['SEASON'] = name 

            team_stats_rookie.append(df)
            print('Download season:', name)
        except Exception as ex:
            print(f"{name} — błąd: {ex}")
            failed_cols.append(name)
        time.sleep(10)

    if len(team_stats_rookie) == len(seasons):
        df_team_rookie = pd.concat(team_stats_rookie, ignore_index=True)
        df_team_rookie.to_csv(path_team_rookie, index=False)
        print('Zapisano do csv')

Zapis dla team rookie
Plik istnieje:../data/raw/team_rookie.csv


In [3]:
import pandas as pd
import time
import os

years = range(2001, 2026) 
path_gs_stat = '../data/raw/gs_stat.csv'
gs_stat_list = []

if os.path.exists(path_gs_stat):
    print(f'Plik istnieje: {path_gs_stat}')
else:
    print('Zapis dla brakującej kolumny GS z Basketball-Reference')
    for year in years:
        try:
            # URL do tabeli per_game dla danego sezonu
            url = f'https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html'
            
            tables = pd.read_html(url)
            df = tables[0].copy()
            
            df = df[df['Player'] != 'Player'].copy()
            
            df['SEASON'] = f'{year-1}-{str(year)[-2:]}'
            
            cols_to_keep = ['Player', 'GS', 'SEASON']
            df_cut = df[cols_to_keep].copy()
            
            df_cut['GS'] = pd.to_numeric(df_cut['GS'], errors='coerce').fillna(0).astype(int)
            
            # Usunięcie duplikatów 
            df_cut = df_cut.drop_duplicates(subset=['Player', 'SEASON'], keep='first')
            
            gs_stat_list.append(df_cut)
            print(f'Pobrano GS dla sezonu: {year}')
            
        except Exception as ex:
            print(f'{year} - błąd pobierania B-Ref: {ex}')

        time.sleep(5)

    if len(gs_stat_list) > 0:
        df_gs = pd.concat(gs_stat_list, ignore_index=True)
        df_gs = df_gs.rename(columns={'Player': 'PLAYER_NAME'})
        df_gs.to_csv(path_gs_stat, index=False)
        print('Zapisano dane GS do csv')

Plik istnieje: ../data/raw/gs_stat.csv
